# Seishin Agent — fine-tune a fast on-device tool-calling model

Trains a small model to drive Seishin's on-device agent: the same 15 tools
(`add_event`, `add_todo`, `add_note`, `list_*`, `update_*`, `delete_*`,
`complete_todo`, `list_inbox`, `remember_entity`, `recall_memory`) and the same
system prompt as `src/services/local-agent.ts`, so it slots straight into the app.

**How to run:** `Runtime → Change runtime type → T4 GPU` (free tier is fine), then `Runtime → Run all`.

1. Generates ~3k synthetic conversations from the app's real tool schemas
2. LoRA fine-tunes **Qwen2.5-0.5B-Instruct** (fast on 4 GB phones) — switch to 1.5B below for higher accuracy
3. Trains with early stopping, then measures tool-call accuracy on held-out data and
   keeps training on fresh data until it reaches the target (or the round limit)
4. Exports `seishin-agent-Q4_K_M.gguf`, smoke-tests it through llama.cpp's own
   tool-call parser (the one llama.rn uses), saves it to Google Drive and downloads it

Then copy the `.gguf` to the phone and pick it in **Settings → On-device model**.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
BASE_MODEL   = "unsloth/Qwen2.5-0.5B-Instruct"   # fastest. Better accuracy: "unsloth/Qwen2.5-1.5B-Instruct"
OUT_NAME     = "seishin-agent"
TRAIN_N      = 3000      # synthetic train conversations per round
EVAL_N       = 300
MAX_EPOCHS   = 3         # per round; early stopping usually ends sooner
TARGET_ACC   = 0.92      # keep training until exact tool-call accuracy reaches this…
MAX_ROUNDS   = 3         # …or this many rounds have run
EVAL_SAMPLES = 200       # conversations scored per accuracy check
QUANT        = "q4_k_m"  # GGUF quantization (q8_0 = bigger, slightly better)
SAVE_TO_DRIVE = True
HF_REPO      = ""        # optional, e.g. "you/seishin-agent-gguf" (needs an HF_TOKEN Colab secret)

In [ ]:
%%capture
!pip install -q unsloth   # pulls matching torch / transformers / trl / peft versions

## 1 · Dataset (generator + tool schemas from the Seishin repo)

In [ ]:
%%writefile tools.json
[
  {
    "type": "function",
    "function": {
      "name": "add_event",
      "description": "Add an event to the calendar",
      "parameters": {
        "type": "object",
        "properties": {
          "title": {
            "type": "string",
            "description": "Event title"
          },
          "startDate": {
            "type": "string",
            "description": "Start date/time in ISO format (e.g. 2024-03-15T14:00:00.000Z)"
          },
          "endDate": {
            "type": "string",
            "description": "End date/time in ISO format"
          },
          "description": {
            "type": "string",
            "description": "Optional event description"
          },
          "notes": {
            "type": "string",
            "description": "Optional freeform notes for the event"
          }
        },
        "required": [
          "title",
          "startDate"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_events",
      "description": "List all calendar events",
      "parameters": {
        "type": "object",
        "properties": {},
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_inbox",
      "description": "List recent notifications/emails/chats in your inbox. Use this to see what notifications you've received.",
      "parameters": {
        "type": "object",
        "properties": {
          "limit": {
            "type": "number",
            "description": "Max items to return (default 20)"
          },
          "unreadOnly": {
            "type": "boolean",
            "description": "Only show unread items"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "update_event",
      "description": "Edit an existing calendar event. Identify it by id or by its current title (query), then change any of title, start/end time, or description.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Current title (or part of it) of the event to edit"
          },
          "id": {
            "type": "string",
            "description": "Exact event id, if known"
          },
          "title": {
            "type": "string",
            "description": "New title"
          },
          "startDate": {
            "type": "string",
            "description": "New start date/time in ISO format"
          },
          "endDate": {
            "type": "string",
            "description": "New end date/time in ISO format"
          },
          "description": {
            "type": "string",
            "description": "New description"
          },
          "notes": {
            "type": "string",
            "description": "New freeform notes for the event"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "delete_event",
      "description": "Delete/remove a calendar event, identified by id or by its title (query).",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Title (or part of it) of the event to delete"
          },
          "id": {
            "type": "string",
            "description": "Exact event id, if known"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "add_todo",
      "description": "Add a task with an optional due date, a checklist of items to do/answer, and reminders. Use reminder \"persistent\" when the user wants to be reminded until it's done (e.g. homework, bills).",
      "parameters": {
        "type": "object",
        "properties": {
          "title": {
            "type": "string",
            "description": "Task title"
          },
          "priority": {
            "type": "string",
            "description": "Priority: low, medium, or high"
          },
          "dueDate": {
            "type": "string",
            "description": "Optional due date/time in ISO format (local time)"
          },
          "reminder": {
            "type": "string",
            "description": "persistent (keep reminding until done), normal (before the deadline), or off"
          },
          "items": {
            "type": "array",
            "items": {
              "type": "string"
            },
            "description": "Optional checklist: questions or steps, as strings"
          }
        },
        "required": [
          "title"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_todos",
      "description": "List tasks, soonest deadline first, with overdue ones flagged, checklist progress and where they came from (Todoist, Notion, a course feed). By default only open tasks.",
      "parameters": {
        "type": "object",
        "properties": {
          "includeCompleted": {
            "type": "boolean",
            "description": "Include completed tasks"
          },
          "withinDays": {
            "type": "number",
            "description": "Only tasks due within this many days (overdue always included)"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "complete_todo",
      "description": "Mark a todo as done (or reopen it), identified by id or by its title (query).",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Title (or part of it) of the todo"
          },
          "id": {
            "type": "string",
            "description": "Exact todo id, if known"
          },
          "done": {
            "type": "boolean",
            "description": "true to mark done (default), false to reopen it"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "update_todo",
      "description": "Edit an existing todo. Identify it by id or by its current title (query), then change title, priority, or due date.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Current title (or part of it) of the todo to edit"
          },
          "id": {
            "type": "string",
            "description": "Exact todo id, if known"
          },
          "title": {
            "type": "string",
            "description": "New title"
          },
          "priority": {
            "type": "string",
            "description": "New priority: low, medium, or high"
          },
          "dueDate": {
            "type": "string",
            "description": "New due date in ISO format"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "delete_todo",
      "description": "Delete/remove a todo, identified by id or by its title (query).",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Title (or part of it) of the todo to delete"
          },
          "id": {
            "type": "string",
            "description": "Exact todo id, if known"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "add_note",
      "description": "Create a note in the note-taking app. Use for anything the user wants to jot down, save, or remember as free text (ideas, lists, meeting notes, thoughts) — as opposed to a dated task (add_todo) or a scheduled event (add_event).",
      "parameters": {
        "type": "object",
        "properties": {
          "title": {
            "type": "string",
            "description": "Short note title"
          },
          "body": {
            "type": "string",
            "description": "The note content / body text"
          },
          "tags": {
            "type": "string",
            "description": "Comma-separated tags, e.g. 'work,ideas'"
          },
          "pinned": {
            "type": "boolean",
            "description": "Pin the note to the top"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_notes",
      "description": "List saved notes (output includes each note's id). Optionally filter by a search query matching title, body, or tags.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Optional text to filter notes by"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "update_note",
      "description": "Edit an existing note. Identify it by id or by its current title (query), then change title, body, tags, or pin.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Current title (or part of it) of the note to edit"
          },
          "id": {
            "type": "string",
            "description": "Exact note id, if known"
          },
          "title": {
            "type": "string",
            "description": "New title"
          },
          "body": {
            "type": "string",
            "description": "New body text (replaces the existing body)"
          },
          "tags": {
            "type": "string",
            "description": "New comma-separated tags (replaces existing tags)"
          },
          "pinned": {
            "type": "boolean",
            "description": "Pin (true) or unpin (false) the note"
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "remember_entity",
      "description": "Store an entity in your knowledge graph. Use this when the user tells you something important about a person, place, concept, or thing that they might want you to recall later.",
      "parameters": {
        "type": "object",
        "properties": {
          "name": {
            "type": "string",
            "description": "Entity name"
          },
          "type": {
            "type": "string",
            "description": "Entity type (person, place, concept, project, preference, etc.)"
          },
          "description": {
            "type": "string",
            "description": "What to remember about this entity"
          },
          "tags": {
            "type": "string",
            "description": "Optional comma-separated tags"
          }
        },
        "required": [
          "name",
          "type",
          "description"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "recall_memory",
      "description": "Search your knowledge graph. Call this when the user asks about something they told you earlier, or when you need context from prior conversations.",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "What to search for in the knowledge graph"
          }
        },
        "required": [
          "query"
        ]
      }
    }
  }
]

In [ ]:
%%writefile generate_dataset.py
#!/usr/bin/env python3
"""
Synthetic tool-calling dataset for Seishin's on-device agent.

Every example is a full OpenAI-style chat — the exact system prompt that
`src/services/local-agent.ts` builds, the user turn, the assistant's tool calls,
the tool results in the exact strings the app's tools return, and the short
final reply. Tool schemas come from `tools.json`, dumped from the app's own
`createDefaultTools()` + `LOCAL_TOOL_NAMES`, so the model learns the real API.

    python3 generate_dataset.py --train 3000 --eval 300 --out data/

Deterministic for a given --seed. No dependencies beyond the stdlib.
"""
from __future__ import annotations

import argparse
import json
import random
import string
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

HERE = Path(__file__).resolve().parent
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]

# --------------------------------------------------------------------------- #
# Formatting that mirrors what the app (Hermes, en-US) prints                  #
# --------------------------------------------------------------------------- #

def hm12(d: datetime) -> str:
    """toLocaleTimeString en-US 2-digit: '09:05 AM'."""
    return d.strftime("%I:%M %p")


def system_prompt(now: datetime, known: str = "") -> str:
    """Byte-for-byte the prompt from createLocalSystemPrompt() in local-agent.ts."""
    date = f"{WEEKDAYS[now.weekday()]}, {MONTHS[now.month - 1]} {now.day}, {now.year}"
    context = f"\n\nWhat you know:\n{known[:1500]}" if known.strip() else ""
    return f"""You are Seishin, an assistant running fully on the user's phone, offline.
Today is {date}, {hm12(now)}.
Use the tools to add, read, change or delete the user's events, tasks and notes, and to remember facts about them.
- When the user asks for an action, call the tool right away. Never ask permission.
- "task", "todo", "homework", "deadline" → add_todo. "meeting", "event", "at <time>" → add_event. "note", "write down" → add_note.
- Resolve "tomorrow", "next Friday" etc. from today's date.
- After tools run, reply in one or two short sentences. If no tool fits, just answer.{context}"""


def iso(d: datetime) -> str:
    """Local ISO datetime, no zone — `new Date()` reads it as phone-local time."""
    return d.strftime("%Y-%m-%dT%H:%M:%S")


def locale_string(d: datetime) -> str:
    """new Date().toLocaleString(): '11/12/2026, 3:00:00 PM'."""
    return f"{d.month}/{d.day}/{d.year}, {d.hour % 12 or 12}:{d.minute:02d}:00 {'AM' if d.hour < 12 else 'PM'}"


def short_due(d: datetime) -> str:
    """toLocaleString([], {weekday:'short', month:'short', day, hour, minute}): 'Thu, Nov 12, 3:00 PM'."""
    return f"{WEEKDAYS[d.weekday()][:3]}, {MONTHS[d.month - 1][:3]} {d.day}, {d.hour % 12 or 12}:{d.minute:02d} {'AM' if d.hour < 12 else 'PM'}"


def human_time(d: datetime) -> str:
    return f"{d.hour % 12 or 12}:{d.minute:02d} {'AM' if d.hour < 12 else 'PM'}"


def human_day(d: datetime, now: datetime) -> str:
    delta = (d.date() - now.date()).days
    if delta == 0:
        return "today"
    if delta == 1:
        return "tomorrow"
    return f"{WEEKDAYS[d.weekday()]}, {MONTHS[d.month - 1][:3]} {d.day}"


def app_id(rng: random.Random, prefix: str) -> str:
    """Looks like uid(prefix) from utils/id.ts."""
    t = "".join(rng.choice(string.digits + "abcdefghijklmnopqrstuvwxyz") for _ in range(8))
    r = "".join(rng.choice(string.digits + "abcdefghijklmnopqrstuvwxyz") for _ in range(4))
    return f"{prefix}-{t}-{rng.randint(1, 999):x}-{r}"


# --------------------------------------------------------------------------- #
# Vocabulary                                                                   #
# --------------------------------------------------------------------------- #

PEOPLE = ["Mia", "Josh", "Ana", "Carlo", "Liza", "Ben", "Aiko", "Sam", "Nico", "Jade", "Paolo",
          "Rhea", "Marco", "Kai", "Elena", "Tomas", "Bea", "Leo", "Grace", "Dr. Santos", "Prof. Reyes"]
EVENTS = ["Dentist appointment", "Team standup", "Project sync", "Gym session", "Doctor's checkup",
          "Job interview", "Client call", "Yoga class", "Haircut", "Piano lesson", "Group study",
          "Thesis defense", "Sprint planning", "Car service", "Board meeting", "Parent-teacher meeting",
          "Movie night", "Basketball practice", "Church service", "Flight to Cebu", "Code review",
          "Coffee chat", "Design review", "1:1 with manager", "Vet appointment", "Birthday party",
          "Swimming lesson", "Lab session", "Seminar", "Concert"]
WITH_PERSON = ["Lunch with {p}", "Dinner with {p}", "Call with {p}", "Meeting with {p}",
               "Coffee with {p}", "Study session with {p}", "Date with {p}", "Catch-up with {p}"]
TASKS = ["Math homework", "Pay electricity bill", "Submit lab report", "Renew passport",
         "Buy groceries", "Finish slides", "Email the landlord", "Read chapter 5", "Fix the bike",
         "Pay rent", "Physics problem set", "Update resume", "Book flights", "Clean the room",
         "Water the plants", "Call the bank", "Return library books", "Write essay draft",
         "Review pull request", "Pay internet bill", "Prepare presentation", "History reading",
         "Submit tax forms", "Order birthday gift", "Laundry", "Study for chemistry quiz",
         "Reply to Prof. Reyes", "Backup photos", "Cancel gym membership", "Finish capstone chapter 2"]
CHECKLISTS = {
    "Buy groceries": ["Eggs", "Milk", "Rice", "Bread", "Coffee", "Bananas", "Chicken", "Soap"],
    "Physics problem set": ["Question 1", "Question 2", "Question 3", "Question 4", "Question 5"],
    "Prepare presentation": ["Outline", "Slides", "Speaker notes", "Rehearse"],
    "Write essay draft": ["Outline", "Introduction", "Body", "Conclusion", "References"],
    "Math homework": ["Exercise 1", "Exercise 2", "Exercise 3", "Exercise 4"],
    "Book flights": ["Compare prices", "Pick seats", "Pay"],
}
NOTE_TOPICS = [
    ("App ideas", "A habit tracker that works offline; a shared grocery list over P2P", "ideas"),
    ("Wifi password", "Home: sunflower-2026, Office: ask IT", "home"),
    ("Book recommendations", "Project Hail Mary, Piranesi, The Midnight Library", "books"),
    ("Meeting notes", "Ship v2 by Friday. Josh owns QA, Mia owns release notes.", "work"),
    ("Gift ideas for Mom", "Orchid plant, scarf, cooking class voucher", "family,gifts"),
    ("Workout plan", "Mon: legs, Wed: push, Fri: pull, Sat: run 5k", "fitness"),
    ("Recipe: adobo", "Chicken, soy sauce, vinegar, garlic, bay leaves, pepper. Simmer 40 min.", "recipes"),
    ("Thesis outline", "1. Intro 2. Related work 3. Method 4. Results 5. Discussion", "school"),
    ("Packing list", "Passport, charger, toothbrush, 3 shirts, jacket", "travel"),
    ("Quotes", "Simplicity is prerequisite for reliability.", "quotes"),
    ("Car parking", "Level B2, slot 47 near the elevator", ""),
    ("Movie list", "Dune Part Two, Past Lives, Spirited Away", "movies"),
    ("Budget", "Rent 12000, food 6000, transport 2000, savings 5000", "money"),
    ("Startup notes", "Talk to 10 users before building anything", "ideas,work"),
]
FACTS = [  # (user says, name, type, description)
    ("My sister's name is Ana and she lives in Davao", "Ana", "person", "User's sister; lives in Davao"),
    ("I'm allergic to peanuts", "Peanut allergy", "preference", "User is allergic to peanuts"),
    ("My manager is Josh", "Josh", "person", "User's manager"),
    ("I work at Globe as a data analyst", "Globe", "organization", "User works there as a data analyst"),
    ("I study computer science at UP Diliman", "UP Diliman", "place", "User studies computer science there"),
    ("my birthday is March 14", "User birthday", "event", "User's birthday is March 14"),
    ("I prefer meetings in the morning", "Morning meetings", "preference", "User prefers meetings in the morning"),
    ("My dog's name is Mochi", "Mochi", "pet", "User's dog"),
    ("I'm vegetarian", "Vegetarian", "preference", "User is vegetarian"),
    ("Carlo is my best friend from high school", "Carlo", "person", "User's best friend from high school"),
    ("My capstone project is called Seishin", "Seishin", "project", "User's capstone project"),
    ("I go to the gym at Anytime Fitness in BGC", "Anytime Fitness BGC", "place", "User's gym"),
    ("my girlfriend is Liza", "Liza", "person", "User's girlfriend"),
    ("I usually sleep at 11pm", "Sleep schedule", "preference", "User usually sleeps at 11pm"),
]
SMALL_TALK = [
    ("hi", "Hi! What can I help you with today?"),
    ("hello", "Hello! Need to add something to your calendar or tasks?"),
    ("thanks!", "You're welcome!"),
    ("thank you", "Anytime."),
    ("who are you?", "I'm Seishin, your assistant. I run fully on your phone and can manage your events, tasks and notes."),
    ("what can you do?", "I can add, find, change or delete your events, tasks and notes, check your inbox, and remember things you tell me."),
    ("are you online?", "No — I run entirely on your phone, offline. Your data stays on the device."),
    ("good morning", "Good morning! Want a look at what's on today?"),
    ("tell me a joke", "Why did the calendar break up with the clock? It needed more dates."),
    ("how do I stay focused while studying?", "Try 25-minute focus blocks with 5-minute breaks, and keep your phone out of reach. I can add study blocks to your calendar if you like."),
    ("what's 15% of 2400?", "15% of 2400 is 360."),
    ("convert 5 km to miles", "5 km is about 3.11 miles."),
    ("ok", "Okay! Let me know if you need anything."),
    ("give me a tip for better sleep", "Keep a fixed bedtime and skip screens for the last 30 minutes before sleeping."),
    ("what's the capital of Japan?", "Tokyo."),
    ("is it better to study at night or morning?", "Mornings work best for most people since you're rested, but consistency matters more than the hour."),
    ("salamat", "Walang anuman!"),
    ("kumusta?", "Ayos lang! Ano'ng maitutulong ko?"),
    ("how many days are in a leap year?", "366."),
    ("what's a good way to plan my week?", "List your fixed commitments first, then block time for your top three priorities. I can put those blocks on your calendar."),
    ("can you explain what a habit stack is?", "It's linking a new habit to one you already do, like stretching right after brushing your teeth."),
    ("what should I eat before a workout?", "Something light with carbs and a little protein about an hour before, like a banana with yogurt."),
    ("how long should a power nap be?", "About 10 to 20 minutes, so you wake up before deep sleep."),
    ("what does ASAP mean?", "As soon as possible."),
    ("translate 'good night' to Japanese", "Oyasumi nasai (おやすみなさい)."),
    ("what's 12 times 17?", "204."),
    ("how do I say no to extra work politely?", "Thank them, say your plate is full right now, and offer a later time or an alternative."),
    ("you're helpful", "Thanks! Happy to help."),
    ("never mind", "No problem."),
    ("what's the pomodoro technique?", "Work for 25 minutes, take a 5-minute break, and after four rounds take a longer 15–30 minute break."),
    ("do you store my data in the cloud?", "No. Everything stays on your phone."),
    ("bye", "Bye! Talk soon."),
    ("how far is the moon?", "About 384,000 km from Earth on average."),
    ("give me a motivational quote", "\"It always seems impossible until it's done.\" — Nelson Mandela"),
]
INBOX = [
    ("gmail", "Prof. Reyes", "Quiz moved to Thursday 10 AM"),
    ("messenger", "Mia", "Are we still on for lunch tomorrow?"),
    ("gmail", "Meralco", "Your bill of ₱2,340 is due Nov 20"),
    ("sms", "GCash", "You received ₱500 from Carlo"),
    ("slack", "Josh", "Can you review my PR before 5?"),
    ("gmail", "Shopee", "Your order has shipped"),
    ("viber", "Mom", "Call me when you're free"),
    ("gmail", "HR", "Interview scheduled Monday 2 PM"),
]

# --------------------------------------------------------------------------- #
# Time expressions: (phrase, resolved datetime)                                #
# --------------------------------------------------------------------------- #

def next_weekday(now: datetime, wd: int) -> datetime:
    days = (wd - now.weekday()) % 7 or 7
    return now + timedelta(days=days)


def pick_day(rng: random.Random, now: datetime) -> tuple[str, datetime]:
    r = rng.random()
    if r < 0.18:
        return "today", now
    if r < 0.45:
        return rng.choice(["tomorrow", "tmrw", "tomorrow", "bukas"]), now + timedelta(days=1)
    if r < 0.52:
        return "the day after tomorrow", now + timedelta(days=2)
    if r < 0.75:
        wd = rng.randrange(7)
        while wd == now.weekday():
            wd = rng.randrange(7)
        d = next_weekday(now, wd)
        return rng.choice([f"on {WEEKDAYS[wd]}", WEEKDAYS[wd].lower(), f"this {WEEKDAYS[wd]}", f"next {WEEKDAYS[wd]}"]), d
    if r < 0.85:
        n = rng.randint(2, 10)
        return f"in {n} days", now + timedelta(days=n)
    d = now + timedelta(days=rng.randint(3, 60))
    return rng.choice([f"on {MONTHS[d.month - 1]} {d.day}", f"on {MONTHS[d.month - 1][:3]} {d.day}", f"{d.month}/{d.day}"]), d


def pick_time(rng: random.Random) -> tuple[str, int, int]:
    r = rng.random()
    if r < 0.12:
        return "at noon", 12, 0
    if r < 0.2:
        return rng.choice([("in the morning", 9, 0), ("tonight", 20, 0), ("in the evening", 19, 0), ("in the afternoon", 15, 0)])
    h = rng.choice([7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20])
    m = rng.choice([0, 0, 0, 0, 15, 30, 30, 45])
    h12, ap = h % 12 or 12, "am" if h < 12 else "pm"
    if m == 0:
        txt = rng.choice([f"at {h12}{ap}", f"at {h12} {ap}", f"at {h12}{ap.upper()}", f"{h12}{ap}", f"at {h:02d}:00"])
    else:
        txt = rng.choice([f"at {h12}:{m:02d}{ap}", f"at {h12}:{m:02d} {ap}", f"{h12}:{m:02d}{ap}"])
    return txt, h, m


def when(rng: random.Random, now: datetime) -> tuple[str, datetime]:
    dtxt, day = pick_day(rng, now)
    for _ in range(10):
        ttxt, h, m = pick_time(rng)
        dt = day.replace(hour=h, minute=m, second=0, microsecond=0)
        if dt > now:
            break
    else:
        dtxt, day = "tomorrow", now + timedelta(days=1)
        ttxt, h, m = "at 10am", 10, 0
        dt = day.replace(hour=h, minute=m, second=0, microsecond=0)
    phrase = f"{dtxt} {ttxt}" if rng.random() < 0.7 else f"{ttxt} {dtxt}"
    return phrase, dt


# --------------------------------------------------------------------------- #
# Conversation builder                                                         #
# --------------------------------------------------------------------------- #

@dataclass
class Ctx:
    rng: random.Random
    now: datetime
    n_calls: int = 0

    def call(self, name: str, args: dict) -> dict:
        self.n_calls += 1
        return {"id": f"call_{self.n_calls}", "type": "function", "function": {"name": name, "arguments": args}}


def assistant_calls(*calls: dict) -> dict:
    return {"role": "assistant", "content": "", "tool_calls": list(calls)}


def tool_msg(call: dict, content: str) -> dict:
    return {"role": "tool", "tool_call_id": call["id"], "name": call["function"]["name"], "content": content}


def say(text: str) -> dict:
    return {"role": "assistant", "content": text}


def user(text: str, rng: random.Random) -> dict:
    for a, b in (("to at ", "to "), ("to on ", "to "), ("by on ", "by "), ("due on ", "due "), ("it to in ", "it to ")):
        text = text.replace(a, b)
    if rng.random() < 0.25:
        text = text.lower()
    if rng.random() < 0.3:
        text = text.rstrip(".!?")
    elif rng.random() < 0.2 and not text.endswith(("?", "!", ".")):
        text += rng.choice([".", "!", " pls", " please"])
    return {"role": "user", "content": text}


def cap(s: str) -> str:
    return s[:1].upper() + s[1:]


# --- scenarios: each returns a list of messages after the system prompt ----- #

def s_add_event(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(EVENTS) if rng.random() < 0.6 else rng.choice(WITH_PERSON).format(p=rng.choice(PEOPLE))
    phrase, dt = when(rng, c.now)
    args = {"title": title, "startDate": iso(dt)}
    dur_txt = ""
    if rng.random() < 0.3:
        hours = rng.choice([0.5, 1.5, 2, 3])
        args["endDate"] = iso(dt + timedelta(hours=hours))
        dur_txt = rng.choice([f" for {hours:g} hours" if hours != 0.5 else " for 30 minutes", f" for {int(hours * 60)} mins"])
    extra = ""
    if rng.random() < 0.15:
        loc = rng.choice(["at the clinic on 5th Ave", "in room 302", "on Zoom", "at SM North", "at the office"])
        args["description"] = cap(loc.removeprefix("at ").removeprefix("in ").removeprefix("on ").removeprefix("the "))
        extra = f" {loc}"
    t = title.lower() if rng.random() < 0.5 else title
    templates = [
        f"Add {t} {phrase}{dur_txt}{extra}",
        f"Schedule {t} {phrase}{dur_txt}{extra}",
        f"I have {t} {phrase}{extra}",
        f"put {t} on my calendar {phrase}{dur_txt}",
        f"Remind me about {t} {phrase}{extra}",
        f"Can you add an event: {t}, {phrase}{dur_txt}",
        f"{cap(t)} {phrase}{extra}",
        f"set a meeting — {t} {phrase}{dur_txt}",
        f"Pa-add naman ng {t} {phrase}",
        f"new event {t} {phrase}{extra}",
    ]
    msg = rng.choice(templates)
    if "endDate" in args and dur_txt not in msg:
        args.pop("endDate")
    if "description" in args and extra not in msg:
        args.pop("description")
    call = c.call("add_event", args)
    reply = rng.choice([
        f"Added \"{title}\" {human_day(dt, c.now)} at {human_time(dt)}.",
        f"Done — {title} is on your calendar for {human_day(dt, c.now)} at {human_time(dt)}.",
        f"Scheduled {title} for {human_day(dt, c.now)}, {human_time(dt)}.",
    ])
    return [user(msg, rng), assistant_calls(call),
            tool_msg(call, f"Added event: \"{title}\" on {locale_string(dt)}"), say(reply)]


def s_add_todo(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    args: dict = {"title": title}
    parts = []
    due = None
    if rng.random() < 0.75:
        dtxt, day = pick_day(rng, c.now)
        if rng.random() < 0.5:
            due = day.replace(hour=23, minute=59, second=0, microsecond=0)
            parts.append(rng.choice([f"by {dtxt}".replace("by on ", "by "), f"due {dtxt}".replace("due on ", "due "), f"{dtxt}"]))
        else:
            ttxt, h, m = pick_time(rng)
            due = day.replace(hour=h, minute=m, second=0, microsecond=0)
            if due <= c.now:
                due = due + timedelta(days=1)
                dtxt = "tomorrow"
            parts.append(f"due {dtxt} {ttxt}".replace("due on ", "due "))
        args["dueDate"] = iso(due)
    priority_txt = ""
    if rng.random() < 0.25:
        p = rng.choice(["high", "high", "low"])
        args["priority"] = p
        priority_txt = rng.choice({"high": ["urgent", "important", "high priority"], "low": ["low priority", "not urgent"]}[p])
    persistent = False
    if rng.random() < 0.25:
        persistent = True
        args["reminder"] = "persistent"
    items = None
    if title in CHECKLISTS and rng.random() < 0.6:
        pool = CHECKLISTS[title]
        items = rng.sample(pool, k=rng.randint(2, min(5, len(pool))))
        args["items"] = items
    word = rng.choice(["task", "todo", "to-do", "reminder"]) if "homework" not in title.lower() else "homework"
    when_txt = (" " + parts[0]) if parts else ""
    pr = f" ({priority_txt})" if priority_txt else ""
    keep = rng.choice([" and keep reminding me until it's done", ", remind me until I finish it", " — nag me until it's done"]) if persistent else ""
    lst = (": " + ", ".join(items)) if items else ""
    t = title.lower() if rng.random() < 0.6 else title
    msg = rng.choice([
        f"Add a {word} to {t}{when_txt}{pr}{keep}{lst}",
        f"I need to {t}{when_txt}{pr}{keep}{lst}",
        f"new {word}: {t}{when_txt}{pr}{keep}{lst}",
        f"remind me to {t}{when_txt}{keep}{pr}{lst}",
        f"{cap(t)}{when_txt}{pr}{keep}{lst}",
        f"todo {t}{when_txt}{pr}{lst}{keep}",
    ]).replace(" to to ", " to ")
    call = c.call("add_todo", args)
    res = f"Added task \"{title}\""
    if due:
        res += f" due {short_due(due)}"
    if items:
        res += f" with {len(items)} checklist item{'s' if len(items) > 1 else ''}"
    if persistent:
        res += ". I'll keep reminding until it's done"
    res += "."
    reply = f"Added \"{title}\"" + (f", due {human_day(due, c.now)} at {human_time(due)}" if due else "") + "."
    if persistent:
        reply += " I'll keep reminding you until it's done."
    elif items:
        reply += f" It has {len(items)} checklist items."
    return [user(msg, rng), assistant_calls(call), tool_msg(call, res), say(reply)]


def s_add_note(c: Ctx) -> list[dict]:
    rng = c.rng
    title, body, tags = rng.choice(NOTE_TOPICS)
    args: dict = {"title": title, "body": body}
    msg = rng.choice([
        f"Write down: {body}",
        f"Note this — {title.lower()}: {body}",
        f"Save a note called {title}: {body}",
        f"take a note: {body}",
        f"jot down my {title.lower()}: {body}",
        f"Make a note \"{title}\" — {body}",
    ])
    if tags and rng.random() < 0.4:
        args["tags"] = tags
        msg += f" (tag it {tags.replace(',', ' and ')})"
    if rng.random() < 0.15:
        args["pinned"] = True
        msg += " and pin it"
    call = c.call("add_note", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Added note: \"{title}\""),
            say(rng.choice([f"Saved your note \"{title}\".", f"Noted — \"{title}\" is in your notes.", f"Saved \"{title}\"" + (" and pinned it." if args.get("pinned") else "."), ]))]


def fake_events(c: Ctx, k: int) -> list[tuple[str, str, datetime]]:
    rng = c.rng
    out = []
    for _ in range(k):
        _, dt = when(rng, c.now)
        title = rng.choice(EVENTS) if rng.random() < 0.6 else rng.choice(WITH_PERSON).format(p=rng.choice(PEOPLE))
        out.append((app_id(rng, "evt"), title, dt))
    return sorted(out, key=lambda e: e[2])


def event_lines(evts) -> str:
    return "\n".join(f"- [id:{i}] {t} ({d.month}/{d.day}/{d.year} {hm12(d)})" for i, t, d in evts)


def s_list_events(c: Ctx) -> list[dict]:
    rng = c.rng
    evts = fake_events(c, rng.randint(0, 5))
    msg = rng.choice(["What's on my calendar?", "what do I have coming up", "show my events", "any meetings this week?",
                      "What's my schedule?", "do I have anything tomorrow?", "list my events", "Ano schedule ko?"])
    call = c.call("list_events", {})
    if not evts:
        return [user(msg, rng), assistant_calls(call), tool_msg(call, "No events found."),
                say("Your calendar is clear — no events coming up.")]
    first = evts[0]
    summary = "; ".join(f"{t} {human_day(d, c.now)} at {human_time(d)}" for _, t, d in evts[:3])
    reply = (f"You have {len(evts)} event{'s' if len(evts) > 1 else ''}: {summary}"
             + (f", and {len(evts) - 3} more." if len(evts) > 3 else "."))
    if "tomorrow" in msg:
        tm = [e for e in evts if (e[2].date() - c.now.date()).days == 1]
        reply = ("Tomorrow you have " + "; ".join(f"{t} at {human_time(d)}" for _, t, d in tm) + ".") if tm else \
            f"Nothing tomorrow. Next up is {first[1]} {human_day(first[2], c.now)} at {human_time(first[2])}."
    return [user(msg, rng), assistant_calls(call), tool_msg(call, event_lines(evts)), say(reply)]


def fake_todos(c: Ctx, k: int):
    rng = c.rng
    out = []
    for title in rng.sample(TASKS, k):
        due = c.now + timedelta(days=rng.randint(-2, 9), hours=rng.randint(0, 10))
        due = due.replace(minute=rng.choice([0, 30, 59]), second=0, microsecond=0)
        out.append((app_id(rng, "todo"), title, rng.choice(["medium", "medium", "high", "low"]), due))
    return sorted(out, key=lambda t: t[3])


def s_list_todos(c: Ctx) -> list[dict]:
    rng = c.rng
    todos = fake_todos(c, rng.randint(0, 5))
    args: dict = {}
    msg = rng.choice(["What tasks do I have?", "show my todos", "what's due?", "anything overdue?", "my to-do list",
                      "what homework do I have left", "list my tasks"])
    if rng.random() < 0.3:
        n = rng.choice([1, 3, 7])
        args["withinDays"] = n
        msg = rng.choice({1: ["what's due today or tomorrow?", "anything due by tomorrow?"],
                          3: ["what's due in the next 3 days?", "tasks for the next few days"],
                          7: ["what's due this week?", "tasks due within a week"]}[n])
        todos = [t for t in todos if t[3] <= c.now + timedelta(days=n)]
    call = c.call("list_todos", args)
    if not todos:
        return [user(msg, rng), assistant_calls(call), tool_msg(call, "No tasks found."),
                say("You're all caught up — no open tasks.")]
    lines = []
    for i, t, p, d in todos:
        s = f"- [id:{i}] {t} [{p}] due {short_due(d)}"
        if d < c.now:
            s += " — OVERDUE"
        lines.append(s)
    overdue = [t for t in todos if t[3] < c.now]
    reply = f"You have {len(todos)} open task{'s' if len(todos) > 1 else ''}"
    if overdue:
        reply += f"; {', '.join(t[1] for t in overdue)} {'is' if len(overdue) == 1 else 'are'} overdue"
    nxt = [t for t in todos if t[3] >= c.now]
    if nxt:
        reply += f". Next up: {nxt[0][1]}, due {human_day(nxt[0][3], c.now)} at {human_time(nxt[0][3])}"
    return [user(msg, rng), assistant_calls(call), tool_msg(call, "\n".join(lines)), say(reply + ".")]


def s_complete_todo(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    q = title.lower() if rng.random() < 0.5 else " ".join(title.lower().split()[:2])
    done = rng.random() < 0.85
    if done:
        msg = rng.choice([f"I finished {q}", f"mark {q} as done", f"done with {q}", f"check off {q}", f"tapos na {q}"])
        args = {"query": q}
    else:
        msg = rng.choice([f"reopen {q}", f"mark {q} as not done", f"I'm not actually done with {q}"])
        args = {"query": q, "done": False}
    call = c.call("complete_todo", args)
    return [user(msg, rng), assistant_calls(call),
            tool_msg(call, f"Marked todo \"{title}\" as {'done' if done else 'not done'}."),
            say(rng.choice([f"Nice work — \"{title}\" is done.", f"Checked off \"{title}\"."]) if done else f"Reopened \"{title}\".")]


def s_update_event(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(EVENTS)
    q = title.lower() if rng.random() < 0.5 else title.split()[0].lower()
    phrase, dt = when(rng, c.now)
    kind = rng.random()
    if kind < 0.6:
        msg = rng.choice([f"Move my {q} to {phrase}", f"reschedule {q} to {phrase}", f"change {q} to {phrase}",
                          f"push the {q} to {phrase}"])
        args = {"query": q, "startDate": iso(dt)}
        reply = f"Moved \"{title}\" to {human_day(dt, c.now)} at {human_time(dt)}."
        new_title = title
    elif kind < 0.8:
        new_title = rng.choice(EVENTS)
        msg = f"rename {q} to {new_title}"
        args = {"query": q, "title": new_title}
        reply = f"Renamed it to \"{new_title}\"."
    else:
        note = rng.choice(["Bring the documents", "Room changed to 401", "Wear formal attire", "Bring laptop"])
        msg = f"add a note to my {q}: {note.lower()}"
        args = {"query": q, "notes": note}
        reply = f"Added that note to \"{title}\"."
        new_title = title
    call = c.call("update_event", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated event \"{new_title}\"."), say(reply)]


def s_update_todo(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    q = title.lower()
    if rng.random() < 0.5:
        dtxt, day = pick_day(rng, c.now)
        due = day.replace(hour=23, minute=59, second=0, microsecond=0)
        msg = rng.choice([f"push {q} to {dtxt}", f"change the deadline of {q} to {dtxt}", f"{q} is now due {dtxt}"]).replace("to on ", "to ").replace("due on ", "due ")
        args = {"query": q, "dueDate": iso(due)}
        reply = f"\"{title}\" is now due {human_day(due, c.now)}."
    else:
        p = rng.choice(["high", "low"])
        msg = rng.choice([f"make {q} {p} priority", f"set {q} priority to {p}", f"{q} is {'urgent' if p == 'high' else 'not urgent'}"])
        args = {"query": q, "priority": p}
        reply = f"Set \"{title}\" to {p} priority."
    call = c.call("update_todo", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated todo \"{title}\"."), say(reply)]


def s_delete(c: Ctx) -> list[dict]:
    rng = c.rng
    kind = rng.choice(["event", "todo"])
    title = rng.choice(EVENTS if kind == "event" else TASKS)
    q = title.lower()
    msg = rng.choice([f"delete {q}", f"cancel my {q}", f"remove {q} from my {'calendar' if kind == 'event' else 'tasks'}",
                      f"I don't need {q} anymore, delete it", f"pakitanggal {q}"])
    call = c.call(f"delete_{kind}", {"query": q})
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Deleted {kind} \"{title}\"."),
            say(rng.choice([f"Deleted \"{title}\".", f"Removed \"{title}\"."]))]


def s_ambiguous(c: Ctx) -> list[dict]:
    """Tool says several items match → the model asks, it does not guess."""
    rng = c.rng
    p1, p2 = rng.sample(PEOPLE, 2)
    msg = rng.choice(["delete my lunch", "cancel lunch", "move lunch to 2pm"])
    tool = "update_event" if "move" in msg else "delete_event"
    args = {"query": "lunch"}
    if tool == "update_event":
        _, dt = when(rng, c.now)
        dt = dt.replace(hour=14, minute=0)
        args["startDate"] = iso(dt)
    call = c.call(tool, args)
    return [user(msg, rng), assistant_calls(call),
            tool_msg(call, f"Error: Multiple events match \"lunch\": Lunch with {p1}, Lunch with {p2}. Ask which one or pass its id."),
            say(f"You have two lunches — with {p1} and with {p2}. Which one?")]


def s_list_then_act(c: Ctx) -> list[dict]:
    """Multi-step: read first, then act on the id found."""
    rng = c.rng
    evts = fake_events(c, rng.randint(2, 4))
    target = rng.choice(evts)
    msg = rng.choice(["cancel my next meeting", "delete the first event on my calendar", "move my next event an hour later"])
    if "next" in msg or "first" in msg:
        target = evts[0]
    c1 = c.call("list_events", {})
    out = [user(msg, rng), assistant_calls(c1), tool_msg(c1, event_lines(evts))]
    if "move" in msg:
        nd = target[2] + timedelta(hours=1)
        c2 = c.call("update_event", {"id": target[0], "startDate": iso(nd)})
        out += [assistant_calls(c2), tool_msg(c2, f"Updated event \"{target[1]}\"."),
                say(f"Moved \"{target[1]}\" to {human_time(nd)} {human_day(nd, c.now)}.")]
    else:
        c2 = c.call("delete_event", {"id": target[0]})
        out += [assistant_calls(c2), tool_msg(c2, f"Deleted event \"{target[1]}\"."),
                say(f"Cancelled \"{target[1]}\" ({human_day(target[2], c.now)} at {human_time(target[2])}).")]
    return out


def s_multi(c: Ctx) -> list[dict]:
    """Two actions in one message → two parallel tool calls."""
    rng = c.rng
    title = rng.choice(EVENTS)
    phrase, dt = when(rng, c.now)
    task = rng.choice(TASKS)
    dtxt, day = pick_day(rng, c.now)
    due = day.replace(hour=23, minute=59, second=0, microsecond=0)
    msg = rng.choice([f"add {title.lower()} {phrase} and a task to {task.lower()} by {dtxt}",
                      f"Schedule {title.lower()} {phrase}, also remind me to {task.lower()} by {dtxt}"]).replace("by on ", "by ")
    c1 = c.call("add_event", {"title": title, "startDate": iso(dt)})
    c2 = c.call("add_todo", {"title": task, "dueDate": iso(due)})
    return [user(msg, rng), assistant_calls(c1, c2),
            tool_msg(c1, f"Added event: \"{title}\" on {locale_string(dt)}"),
            tool_msg(c2, f"Added task \"{task}\" due {short_due(due)}."),
            say(f"Added {title} {human_day(dt, c.now)} at {human_time(dt)}, and \"{task}\" due {human_day(due, c.now)}.")]


def s_remember(c: Ctx) -> list[dict]:
    rng = c.rng
    said, name, typ, desc = rng.choice(FACTS)
    msg = rng.choice([said, f"remember that {said[0].lower() + said[1:]}", f"fyi {said[0].lower() + said[1:]}"])
    call = c.call("remember_entity", {"name": name, "type": typ, "description": desc})
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Remembered \"{name}\" as a {typ}."),
            say(rng.choice(["Got it, I'll remember that.", "Noted — I'll remember.", f"I'll remember {name}."]))]


def s_recall(c: Ctx) -> list[dict]:
    rng = c.rng
    said, name, typ, desc = rng.choice(FACTS)
    qs = {"person": [f"who is {name}?", f"what do you know about {name}"],
          "pet": ["what's my dog's name?"], "place": [f"what did I tell you about {name}?"]}
    msg = rng.choice(qs.get(typ, [f"what do you remember about {name.lower()}?", f"what did I say about {name.lower()}?"]))
    call = c.call("recall_memory", {"query": name})
    hit = rng.random() < 0.85
    res = f"- **{name}** [id:{app_id(rng, 'ent')}] ({typ}): {desc}" if hit else f"No matches for \"{name}\"."
    reply = f"{desc.replace('User', 'You').replace(' user', ' you')}." if hit else f"I don't have anything saved about {name} yet. Want to tell me?"
    return [user(msg, rng), assistant_calls(call), tool_msg(call, res), say(reply.replace("..", "."))]


def s_inbox(c: Ctx) -> list[dict]:
    rng = c.rng
    items = rng.sample(INBOX, rng.randint(1, 4))
    unread = rng.random() < 0.5
    args = {"unreadOnly": True} if unread else {}
    msg = rng.choice(["any new messages?", "check my unread notifications"] if unread else
                     ["what's in my inbox?", "show my notifications", "did anyone message me?"])
    call = c.call("list_inbox", args)
    d = c.now - timedelta(days=rng.randint(0, 2))
    lines = "\n".join(f"- [{s}] {t}: {b} ({d.month}/{d.day}/{d.year}) [unread]" for s, t, b in items)
    reply = f"You have {len(items)} {'unread ' if unread else ''}item{'s' if len(items) > 1 else ''}: " + \
        "; ".join(f"{t} — {b}" for _, t, b in items[:3]) + "."
    return [user(msg, rng), assistant_calls(call), tool_msg(call, lines), say(reply)]


def s_list_notes(c: Ctx) -> list[dict]:
    rng = c.rng
    title, body, tags = rng.choice(NOTE_TOPICS)
    key = title.split()[0].lower()
    msg = rng.choice([f"find my note about {key}", f"what did I write about {key}?", f"show my {title.lower()} note"])
    call = c.call("list_notes", {"query": key})
    nid = app_id(rng, "note")
    tg = f" #{' #'.join(tags.split(','))}" if tags else ""
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"- [id:{nid}] {title} — {body[:60]}{tg}"),
            say(f"Here it is — \"{title}\": {body}")]


def s_update_note(c: Ctx) -> list[dict]:
    rng = c.rng
    title, body, _ = rng.choice(NOTE_TOPICS)
    if rng.random() < 0.5:
        msg, args, reply = f"pin my {title.lower()} note", {"query": title.lower(), "pinned": True}, f"Pinned \"{title}\"."
    else:
        new = rng.choice(["Updated: ", "v2: "]) + body
        msg = f"replace my {title.lower()} note with: {new}"
        args, reply = {"query": title.lower(), "body": new}, f"Updated \"{title}\"."
    call = c.call("update_note", args)
    return [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated note \"{title}\"."), say(reply)]


def s_tool_error(c: Ctx) -> list[dict]:
    rng = c.rng
    title = rng.choice(TASKS)
    q = title.lower()
    call = c.call("complete_todo", {"query": q})
    return [user(f"mark {q} done", rng), assistant_calls(call),
            tool_msg(call, f"Error: No todo matches \"{q}\"."),
            say(f"I couldn't find a task called \"{title}\". Want me to add it?")]


def s_chat(c: Ctx) -> list[dict]:
    rng = c.rng
    q, a = rng.choice(SMALL_TALK)
    return [user(q, rng), say(a)]


def s_date_question(c: Ctx) -> list[dict]:
    rng = c.rng
    if rng.random() < 0.5:
        return [user(rng.choice(["what day is it today?", "what's the date?", "anong araw ngayon?"]), rng),
                say(f"Today is {WEEKDAYS[c.now.weekday()]}, {MONTHS[c.now.month - 1]} {c.now.day}, {c.now.year}.")]
    wd = rng.randrange(7)
    d = next_weekday(c.now, wd)
    return [user(f"what's the date this coming {WEEKDAYS[wd]}?", rng),
            say(f"This coming {WEEKDAYS[wd]} is {MONTHS[d.month - 1]} {d.day}.")]


def s_followup(c: Ctx) -> list[dict]:
    """Multi-turn: an earlier exchange in history, then a follow-up that refers back."""
    first = s_add_event(c)
    title = first[1]["tool_calls"][0]["function"]["arguments"]["title"]
    rng = c.rng
    phrase, dt = when(rng, c.now)
    msg = rng.choice([f"actually make it {phrase}", f"move it to {phrase}", f"wait, change that to {phrase}"])
    call = c.call("update_event", {"query": title, "startDate": iso(dt)})
    return first + [user(msg, rng), assistant_calls(call), tool_msg(call, f"Updated event \"{title}\"."),
                    say(f"Moved \"{title}\" to {human_day(dt, c.now)} at {human_time(dt)}.")]


SCENARIOS = [  # (weight, fn)
    (16, s_add_event), (16, s_add_todo), (8, s_add_note), (7, s_list_events), (7, s_list_todos),
    (5, s_complete_todo), (6, s_update_event), (4, s_update_todo), (5, s_delete), (2, s_ambiguous),
    (4, s_list_then_act), (4, s_multi), (4, s_remember), (3, s_recall), (3, s_inbox), (3, s_list_notes),
    (2, s_update_note), (2, s_tool_error), (7, s_chat), (2, s_date_question), (4, s_followup),
]


def known_context(rng: random.Random) -> str:
    """Sometimes the app prepends retrieved memory to the prompt."""
    if rng.random() > 0.25:
        return ""
    facts = rng.sample(FACTS, rng.randint(1, 3))
    return "\n".join(f"- {n} ({t}): {d}" for _, n, t, d in facts)


def make_example(rng: random.Random, tools: list[dict]) -> dict:
    now = datetime(2026, 1, 1) + timedelta(days=rng.randint(0, 729), hours=rng.randint(6, 22), minutes=rng.randint(0, 59))
    c = Ctx(rng, now)
    fn = rng.choices([f for _, f in SCENARIOS], weights=[w for w, _ in SCENARIOS])[0]
    msgs = fn(c)
    return {"scenario": fn.__name__.removeprefix("s_"),
            "messages": [{"role": "system", "content": system_prompt(now, known_context(rng))}] + msgs,
            "tools": tools}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", type=int, default=3000)
    ap.add_argument("--eval", type=int, default=300)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--tools", default=str(HERE / "tools.json"))
    ap.add_argument("--out", default=str(HERE / "data"))
    a = ap.parse_args()
    tools = json.loads(Path(a.tools).read_text())
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    seen: set[str] = set()
    for split, n, seed in (("train", a.train, a.seed), ("eval", a.eval, a.seed + 1000)):
        rng = random.Random(seed)
        rows = []
        while len(rows) < n:
            ex = make_example(rng, tools)
            key = json.dumps(ex["messages"][1:], sort_keys=True)
            if key in seen:  # keep eval disjoint from train
                continue
            seen.add(key)
            rows.append(ex)
        with open(out / f"{split}.jsonl", "w") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"{split}: {len(rows)} examples → {out / f'{split}.jsonl'}")


if __name__ == "__main__":
    main()

In [ ]:
!python3 generate_dataset.py --train $TRAIN_N --eval $EVAL_N --seed 7 --out data
import json
train_rows = [json.loads(l) for l in open("data/train.jsonl")]
eval_rows  = [json.loads(l) for l in open("data/eval.jsonl")]
print(json.dumps(train_rows[0]["messages"][1:], indent=1, ensure_ascii=False)[:1500])

## 2 · Model

In [ ]:
import torch
from unsloth import FastLanguageModel
MAX_LEN = 4096  # the app runs llama.rn with n_ctx 4096
model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, dtype=None, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=7,
)

In [ ]:
from datasets import Dataset

def render(row, upto=None, gen_prompt=False):
    # Render in plain Python (not via datasets.map): Arrow would merge every tool-call's
    # arguments into one struct and fill missing keys with nulls.
    msgs = row["messages"] if upto is None else row["messages"][:upto]
    return tokenizer.apply_chat_template(msgs, tools=row["tools"], tokenize=False, add_generation_prompt=gen_prompt)

def to_dataset(rows):
    texts = [render(r) for r in rows]
    n = [len(tokenizer(t).input_ids) for t in texts[:50]]
    print(f"{len(texts)} examples, ~{sum(n)//len(n)} tokens each (max of sample {max(n)})")
    return Dataset.from_dict({"text": texts})

train_ds, eval_ds = to_dataset(train_rows), to_dataset(eval_rows)
print(train_ds[0]["text"][-1200:])

## 3 · Accuracy check (generates with the model and parses `<tool_call>` blocks)

In [ ]:
import re

CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def parse_calls(text):
    out = []
    for m in CALL_RE.finditer(text):
        try:
            c = json.loads(m.group(1)); out.append((c.get("name"), c.get("arguments") or {}))
        except json.JSONDecodeError:
            out.append(("<bad json>", {}))
    return out

def norm(v):
    return v.strip().lower() if isinstance(v, str) else v

def same_args(a, b):
    return isinstance(a, dict) and set(a) == set(b) and all(norm(a[k]) == norm(b[k]) for k in b)

@torch.no_grad()
def evaluate(rows, n=EVAL_SAMPLES, batch=16, show=3):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    cases = []  # every assistant turn of each conversation, given the history before it
    for r in rows[:n]:
        for i, m in enumerate(r["messages"]):
            if m["role"] == "assistant":
                cases.append((render(r, upto=i, gen_prompt=True), m, r["scenario"]))
    stats = dict(turns=0, name_ok=0, exact=0, no_tool_ok=0, no_tool=0, tool_turns=0)
    misses = []
    for s in range(0, len(cases), batch):
        chunk = cases[s:s + batch]
        enc = tokenizer([c[0] for c in chunk], return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=192, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        for (prompt, want, scen), g in zip(chunk, gen):
            text = tokenizer.decode(g[enc.input_ids.shape[1]:], skip_special_tokens=False)
            text = text.split("<|im_end|>")[0].replace("<|endoftext|>", "")
            got = parse_calls(text)
            exp = [(c["function"]["name"], c["function"]["arguments"]) for c in want.get("tool_calls", [])]
            stats["turns"] += 1
            if not exp:
                stats["no_tool"] += 1; ok = not got; stats["no_tool_ok"] += ok
            else:
                stats["tool_turns"] += 1
                names = [g_[0] for g_ in got] == [e[0] for e in exp]; stats["name_ok"] += names
                ok = names and all(same_args(g_[1], e[1]) for g_, e in zip(got, exp)); stats["exact"] += ok
            if not ok and len(misses) < show:
                misses.append((scen, exp or want["content"], text.strip()[:300]))
    tt, nt = max(stats["tool_turns"], 1), max(stats["no_tool"], 1)
    res = {"tool_name_acc": stats["name_ok"] / tt, "tool_exact_acc": stats["exact"] / tt,
           "no_tool_acc": stats["no_tool_ok"] / nt, "turns": stats["turns"]}
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in res.items()})
    for scen, exp, got in misses:
        print(f"  ✗ [{scen}] expected {exp}\n    got {got!r}")
    FastLanguageModel.for_training(model)
    tokenizer.padding_side = "right"
    return res

print("Base model, before training:")
base_scores = evaluate(eval_rows, n=60)

## 4 · Train until done
Early stopping on eval loss within each round; extra rounds on fresh data until the accuracy target is hit.

In [ ]:
from dataclasses import fields
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only
import subprocess

_cfg_fields = {f.name for f in fields(SFTConfig)}
LEN_ARG = "max_seq_length" if "max_seq_length" in _cfg_fields else "max_length"  # renamed in newer TRL

def make_trainer(train, evals, epochs, lr):
    t = SFTTrainer(
        model=model, processing_class=tokenizer, train_dataset=train, eval_dataset=evals,
        args=SFTConfig(
            dataset_text_field="text", packing=False, **{LEN_ARG: MAX_LEN},
            per_device_train_batch_size=4, gradient_accumulation_steps=4,
            num_train_epochs=epochs, learning_rate=lr, lr_scheduler_type="cosine", warmup_ratio=0.05,
            weight_decay=0.01, optim="adamw_8bit", seed=7,
            fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10, eval_strategy="steps", eval_steps=40, save_strategy="steps", save_steps=40,
            save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss",
            per_device_eval_batch_size=4, output_dir="checkpoints", report_to="none",
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
    )
    # Learn only the assistant turns; system, user and tool results (rendered as user) are masked.
    return train_on_responses_only(t, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

history = []
for rnd in range(1, MAX_ROUNDS + 1):
    if rnd > 1:  # fresh, unseen conversations for the next round
        subprocess.run(["python3", "generate_dataset.py", "--train", str(TRAIN_N), "--eval", "50",
                        "--seed", str(100 + rnd), "--out", f"data/r{rnd}"], check=True)
        train_ds = to_dataset([json.loads(l) for l in open(f"data/r{rnd}/train.jsonl")])
    print(f"\n══ Round {rnd} ══")
    trainer = make_trainer(train_ds, eval_ds, MAX_EPOCHS if rnd == 1 else 1, 2e-4 if rnd == 1 else 5e-5)
    trainer.train()
    scores = evaluate(eval_rows)
    history.append(scores)
    if scores["tool_exact_acc"] >= TARGET_ACC and scores["no_tool_acc"] >= 0.85:
        print(f"Target reached in round {rnd}."); break
else:
    print("Round limit reached — best scores:", history[-1], "(raise MAX_ROUNDS or use the 1.5B model)")

In [ ]:
# Try it yourself
def ask(text, now_row=eval_rows[0]):
    row = {"messages": [now_row["messages"][0], {"role": "user", "content": text}], "tools": now_row["tools"]}
    FastLanguageModel.for_inference(model); tokenizer.padding_side = "left"
    enc = tokenizer([render(row, gen_prompt=True)], return_tensors="pt").to("cuda")
    out = model.generate(**enc, max_new_tokens=192, do_sample=False)
    FastLanguageModel.for_training(model); tokenizer.padding_side = "right"
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=False).split("<|im_end|>")[0]

print(eval_rows[0]["messages"][0]["content"].splitlines()[1])
for q in ["add dentist tomorrow at 3pm", "remind me to pay rent by friday, keep reminding me",
          "what's on my calendar?", "write down: buy a new charger", "thanks!"]:
    print(f"> {q}\n{ask(q)}\n")

## 5 · Export GGUF for llama.rn

In [ ]:
import glob, os, shutil
model.save_pretrained(f"{OUT_NAME}-lora"); tokenizer.save_pretrained(f"{OUT_NAME}-lora")
model.save_pretrained_gguf(f"{OUT_NAME}-gguf", tokenizer, quantization_method=QUANT)
ggufs = sorted(glob.glob(f"{OUT_NAME}-gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf"), key=os.path.getsize)
src = [g for g in ggufs if QUANT.lower() in g.lower()] or ggufs
GGUF = f"{OUT_NAME}-{QUANT.upper()}.gguf"
shutil.copy(src[-1], GGUF)
print(GGUF, f"{os.path.getsize(GGUF) / 1e6:.0f} MB")

### Smoke test through llama.cpp's own tool-call parser (`--jinja`, same path as llama.rn)

In [ ]:
import subprocess, time, requests
srv = next((p for p in glob.glob("llama.cpp/**/llama-server", recursive=True) if os.access(p, os.X_OK)), None)
if not srv:
    print("llama-server not built by this unsloth version — skipping (the HF-side accuracy check above still applies).")
else:
    proc = subprocess.Popen([srv, "-m", GGUF, "--jinja", "-c", "4096", "--port", "8088", "-ngl", "99"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        for _ in range(60):
            try:
                if requests.get("http://127.0.0.1:8088/health", timeout=1).ok: break
            except requests.RequestException: pass
            time.sleep(1)
        tools = json.load(open("tools.json"))
        for q in ["add team standup tomorrow at 9am", "what tasks do I have?", "hello!"]:
            r = requests.post("http://127.0.0.1:8088/v1/chat/completions", json={
                "messages": [eval_rows[0]["messages"][0], {"role": "user", "content": q}],
                "tools": tools, "tool_choice": "auto", "temperature": 0.3}).json()
            m = r["choices"][0]["message"]
            print(f"> {q}\n  tool_calls={[(c['function']['name'], c['function']['arguments']) for c in m.get('tool_calls') or []]}"
                  f"\n  content={m.get('content')!r}")
    finally:
        proc.terminate()

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/seishin", exist_ok=True)
    shutil.copy(GGUF, "/content/drive/MyDrive/seishin/")
    shutil.copytree(f"{OUT_NAME}-lora", f"/content/drive/MyDrive/seishin/{OUT_NAME}-lora", dirs_exist_ok=True)
    print("Saved to Drive: MyDrive/seishin/" + GGUF)
if HF_REPO:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_file(path_or_fileobj=GGUF, path_in_repo=GGUF, repo_id=HF_REPO)
    print(f"https://huggingface.co/{HF_REPO}/resolve/main/{GGUF}")
from google.colab import files
files.download(GGUF)

## On the phone
Copy `seishin-agent-Q4_K_M.gguf` to the device → Seishin **Settings → On-device model** → pick the file.
The model's chat template is embedded in the GGUF, so llama.rn's `jinja: true` path renders the
tools and parses `<tool_call>` replies with no app changes.